In [2]:
import requests
import pymysql

# =========================
# 1. 설정
# =========================
API_KEY = "09c879bd95754f1299e6".strip()
API_URL = f"http://openapi.foodsafetykorea.go.kr/api/{API_KEY}/COOKRCP01/json/1/100"

DB_CONFIG = {
    "host": "localhost",      # 👉 네 MySQL 주소
    "user": "root",           # 👉 계정
    "password": "root",  # 👉 비번
    "db": "recipe_app",       # 👉 네가 만든 DB 이름
    "charset": "utf8mb4",
    "cursorclass": pymysql.cursors.DictCursor,
    "autocommit": False,
}


# =========================
# 2. 재료 문자열 파서
#    "돼지고기 200g, 양파 1/2개" → [{name, amount, sort_order}]
# =========================
def parse_ingredients(raw: str):
    if not raw:
        return []
    result = []
    # 공공데이터는 보통 콤마(,)로 구분돼있음
    parts = [p.strip() for p in raw.split(",") if p.strip()]
    for idx, part in enumerate(parts, start=1):
        tokens = part.split()
        if len(tokens) == 1:
            name = tokens[0]
            amount = None
        else:
            name = " ".join(tokens[:-1])
            amount = tokens[-1]
        result.append(
            {
                "name": name,
                "amount": amount,
                "sort_order": idx,
            }
        )
    return result


# =========================
# 3. DB INSERT 함수들
# =========================

def insert_recipe(cur, row):
    """
    공공데이터 1개 row → recipes 1행
    """
    sql = """
        INSERT INTO recipes (
            author_id, category_id, title, description, thumbnail_url,
            cook_time_min, servings, difficulty, estimated_cost, is_minimal
        ) VALUES (NULL, NULL, %s, %s, %s, NULL, NULL, NULL, NULL, 0)
    """
    title = row.get("RCP_NM")
    # 설명이 딱히 없으니 팁이나 재료문자열을 설명으로 쓰자
    desc = row.get("RCP_NA_TIP") or row.get("RCP_PARTS_DTLS")
    thumb = row.get("ATT_FILE_NO_MAIN")
    cur.execute(sql, (title, desc, thumb))
    return cur.lastrowid


def insert_ingredients(cur, recipe_id, raw_parts):
    """
    파싱된 재료 → recipe_ingredients 여러 행
    """
    ingredients = parse_ingredients(raw_parts)
    if not ingredients:
        return
    sql = """
        INSERT INTO recipe_ingredients (
            recipe_id, name, amount, ingredient_id, sort_order
        ) VALUES (%s, %s, %s, NULL, %s)
    """
    for ing in ingredients:
        cur.execute(sql, (
            recipe_id,
            ing["name"],
            ing["amount"],
            ing["sort_order"],
        ))


def insert_steps(cur, recipe_id, row):
    """
    MANUAL01~MANUAL20 → recipe_steps
    """
    sql = """
        INSERT INTO recipe_steps (
            recipe_id, step_no, content, image_url
        ) VALUES (%s, %s, %s, %s)
    """
    for i in range(1, 21):
        text_key = f"MANUAL{str(i).zfill(2)}"
        img_key = f"MANUAL_IMG{str(i).zfill(2)}"
        content = row.get(text_key)
        image_url = row.get(img_key)
        if content and content.strip():
            cur.execute(sql, (
                recipe_id,
                i,
                content.strip(),
                image_url
            ))


def insert_nutrition(cur, recipe_id, row):
    """
    INFO_ENG, INFO_CAR, INFO_PRO, INFO_FAT, INFO_NA → recipe_nutrition
    """
    sql = """
        INSERT INTO recipe_nutrition (
            recipe_id, basis, calories, carbs, protein, fat, sugar, sodium, fiber, extra
        ) VALUES (%s, 'per_serving', %s, %s, %s, %s, NULL, %s, NULL, NULL)
    """
    calories = row.get("INFO_ENG") or None
    carbs = row.get("INFO_CAR") or None
    protein = row.get("INFO_PRO") or None
    fat = row.get("INFO_FAT") or None
    sodium = row.get("INFO_NA") or None

    # 영양정보가 전부 비어있으면 안 넣어도 됨
    if not any([calories, carbs, protein, fat, sodium]):
        return

    cur.execute(sql, (
        recipe_id,
        calories,
        carbs,
        protein,
        fat,
        sodium
    ))


# =========================
# 4. 메인 로직
# =========================

def main():
    # 1) API 호출
    resp = requests.get(API_URL)

    # 응답 확인
    if resp.status_code != 200:
        print("❌ HTTP 에러:", resp.status_code)
        print(resp.text[:300])
        return

    text = resp.text.strip()
    if not text or not (text.startswith("{") or text.startswith("[")):
        print("❌ JSON이 아닌 응답이 왔음. 내용부터 확인하세요:")
        print(text[:500])
        return

    data = resp.json()

    payload = data.get("COOKRCP01")
    if not payload:
        print("❌ 'COOKRCP01' 키가 없음. 실제 응답:", data)
        return

    rows = payload.get("row") or []
    print(f"✅ API에서 {len(rows)}개 레시피 가져옴")

    # 2) DB 연결
    conn = pymysql.connect(**DB_CONFIG)

    try:
        with conn.cursor() as cur:
            for r in rows:
                # 1. 레시피 본체
                recipe_id = insert_recipe(cur, r)

                # 2. 재료
                insert_ingredients(cur, recipe_id, r.get("RCP_PARTS_DTLS"))

                # 3. 조리순서
                insert_steps(cur, recipe_id, r)

                # 4. 영양정보
                insert_nutrition(cur, recipe_id, r)

            conn.commit()
            print("🎉 모든 레시피 DB 저장 완료")
    except Exception as e:
        conn.rollback()
        print("❌ 중간에 에러나서 롤백했음:", e)
    finally:
        conn.close()


if __name__ == "__main__":
    main()


✅ API에서 100개 레시피 가져옴
🎉 모든 레시피 DB 저장 완료


In [4]:
import requests
import pymysql
import time

API_KEY = "09c879bd95754f1299e6".strip()

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "root",
    "db": "cooking_db",
    "charset": "utf8mb4",
    "cursorclass": pymysql.cursors.DictCursor,
    "autocommit": False,
}

# =========================
# 공통: 한 구간만 가져오기 (1~50 이런 거)
# =========================
def fetch_range(start: int, end: int, max_retry: int = 3):
    """
    COOKRCP01/json/{start}/{end} 구간을 요청.
    503 나오면 max_retry만큼 다시 시도.
    """
    url = f"http://openapi.foodsafetykorea.go.kr/api/{API_KEY}/COOKRCP01/json/{start}/{end}"
    for attempt in range(1, max_retry + 1):
        resp = requests.get(url)

        # 503이나 500 뜨면 재시도
        if resp.status_code in (500, 503):
            print(f"⚠️ {start}~{end} 요청 {attempt}번째 실패 (status={resp.status_code})")
            # 살짝 쉬었다 재요청 (서버 과부하 완화)
            time.sleep(1.5)
            continue

        if resp.status_code != 200:
            print(f"❌ {start}~{end} 요청 실패: {resp.status_code}")
            print(resp.text[:200])
            return None

        text = resp.text.strip()
        if not text.startswith("{") and not text.startswith("["):
            print(f"❌ {start}~{end} 응답이 JSON 아님")
            print(text[:200])
            return None

        data = resp.json()
        payload = data.get("COOKRCP01")
        if not payload:
            print(f"❌ {start}~{end} 응답에 COOKRCP01 없음")
            return None

        rows = payload.get("row") or []
        print(f"✅ {start}~{end} 구간에서 {len(rows)}개 가져옴")
        return rows

    # 여기까지 오면 retry 전부 실패
    print(f"❌ {start}~{end} 구간, {max_retry}회 재시도했지만 실패")
    return None


# =========================
# 재료 파서
# =========================
def parse_ingredients(raw: str):
    if not raw:
        return []
    parts = [p.strip() for p in raw.split(",") if p.strip()]
    result = []
    for idx, part in enumerate(parts, start=1):
        tokens = part.split()
        if len(tokens) == 1:
            name = tokens[0]
            amount = None
        else:
            name = " ".join(tokens[:-1])
            amount = tokens[-1]
        result.append(
            {
                "name": name,
                "amount": amount,
                "sort_order": idx,
            }
        )
    return result


# =========================
# DB INSERT 함수
# =========================
def insert_recipe(cur, row):
    sql = """
        INSERT INTO recipes (
            author_id, category_id, title, description, thumbnail_url,
            cook_time_min, servings, difficulty, estimated_cost, is_minimal
        ) VALUES (NULL, NULL, %s, %s, %s, NULL, NULL, NULL, NULL, 0)
    """
    title = row.get("RCP_NM")
    desc = row.get("RCP_NA_TIP") or row.get("RCP_PARTS_DTLS")
    thumb = row.get("ATT_FILE_NO_MAIN")
    cur.execute(sql, (title, desc, thumb))
    return cur.lastrowid


def insert_ingredients(cur, recipe_id, raw_parts):
    ings = parse_ingredients(raw_parts)
    if not ings:
        return
    sql = """
        INSERT INTO recipe_ingredients (
            recipe_id, name, amount, ingredient_id, sort_order
        ) VALUES (%s, %s, %s, NULL, %s)
    """
    for ing in ings:
        cur.execute(sql, (
            recipe_id,
            ing["name"],
            ing["amount"],
            ing["sort_order"]
        ))


def insert_steps(cur, recipe_id, row):
    sql = """
        INSERT INTO recipe_steps (
            recipe_id, step_no, content, image_url
        ) VALUES (%s, %s, %s, %s)
    """
    for i in range(1, 21):
        txt = row.get(f"MANUAL{str(i).zfill(2)}")
        img = row.get(f"MANUAL_IMG{str(i).zfill(2)}")
        if txt and txt.strip():
            cur.execute(sql, (
                recipe_id,
                i,
                txt.strip(),
                img
            ))


def insert_nutrition(cur, recipe_id, row):
    sql = """
        INSERT INTO recipe_nutrition (
            recipe_id, basis, calories, carbs, protein, fat, sugar, sodium, fiber, extra
        ) VALUES (%s, 'per_serving', %s, %s, %s, %s, NULL, %s, NULL, NULL)
    """
    calories = row.get("INFO_ENG") or None
    carbs    = row.get("INFO_CAR") or None
    protein  = row.get("INFO_PRO") or None
    fat      = row.get("INFO_FAT") or None
    sodium   = row.get("INFO_NA") or None

    if not any([calories, carbs, protein, fat, sodium]):
        return

    cur.execute(sql, (
        recipe_id,
        calories,
        carbs,
        protein,
        fat,
        sodium
    ))


# =========================
# 메인
# =========================
def main():
    # 1) MySQL 연결
    conn = pymysql.connect(**DB_CONFIG)

    # 2) 여러 구간 돌기 (1~50, 51~100, 101~150 ... 이런 식)
    #    일단 200개만 예시로
    ranges = [(1, 50), (51, 100), (101, 150), (151, 200)]

    try:
        with conn.cursor() as cur:
            for (start, end) in ranges:
                rows = fetch_range(start, end)
                if not rows:
                    # 이 구간은 서버가 죽어있으니까 스킵하고 다음으로
                    print(f"↪️ {start}~{end} 구간은 스킵")
                    continue

                for r in rows:
                    # 1. 레시피 본체
                    recipe_id = insert_recipe(cur, r)
                    # 2. 재료
                    insert_ingredients(cur, recipe_id, r.get("RCP_PARTS_DTLS"))
                    # 3. 조리순서
                    insert_steps(cur, recipe_id, r)
                    # 4. 영양정보
                    insert_nutrition(cur, recipe_id, r)

            conn.commit()
            print("🎉 가능한 구간은 전부 DB에 저장 완료")

    except Exception as e:
        conn.rollback()
        print("❌ 에러로 롤백:", e)
    finally:
        conn.close()


if __name__ == "__main__":
    main()


✅ 1~50 구간에서 50개 가져옴
✅ 51~100 구간에서 50개 가져옴
✅ 101~150 구간에서 50개 가져옴
✅ 151~200 구간에서 50개 가져옴
🎉 가능한 구간은 전부 DB에 저장 완료


저장된 위치: c:\Users\alstj\AppData\Local\Programs\Microsoft VS Code
